In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import time
import tracemalloc
from itertools import combinations
from mlxtend.frequent_patterns import apriori, fpgrowth


urls = {
    1000: "https://raw.githubusercontent.com/pakornlee/ml_example/23665225ce5781e8ea782e18829e6108a5a4c92f/transactions_1000_onehot.csv",
    3000: "https://raw.githubusercontent.com/pakornlee/ml_example/23665225ce5781e8ea782e18829e6108a5a4c92f/transactions_3000_onehot.csv",
    5000: "https://raw.githubusercontent.com/pakornlee/ml_example/23665225ce5781e8ea782e18829e6108a5a4c92f/transactions_5000_onehot.csv",
    10000: "https://raw.githubusercontent.com/pakornlee/ml_example/23665225ce5781e8ea782e18829e6108a5a4c92f/transactions_10000_onehot.csv"
}

datasets = {}
for size, url in urls.items():
    print(f"Loading {size} transactions...")
    df = pd.read_csv(url)
    df.columns = df.columns.str.strip()
    df = df.astype(int)
    datasets[size] = df


def measure(func):
    tracemalloc.start()
    start = time.time()
    result = func()
    elapsed = time.time() - start
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, elapsed, peak / 1024 / 1024

# ==============================================
# ALGORITHMS 
# ==============================================

def run_apriori(df, min_sup, max_len=6):
    return apriori(df, min_support=min_sup, use_colnames=True, max_len=max_len)

def run_fp(df, min_sup, max_len=6):
    return fpgrowth(df, min_support=min_sup, use_colnames=True, max_len=max_len)

# ==============================================
# COMPACT (Bitset + Apriori prune + Top-N)
# ==============================================

def run_bitset_pruned_topN(df, min_sup, max_k=6, top_n=20):
    total = len(df)

    bit_data = {}
    for col in df.columns:
        bitstring = ''.join(df[col].astype(int).astype(str))
        bit_data[frozenset([col])] = int(bitstring, 2)

    L = {1: {}}
    for itemset, bit in bit_data.items():
        support = bin(bit).count('1') / total
        if support >= min_sup:
            L[1][itemset] = (bit, support)

    L[1] = dict(sorted(L[1].items(), key=lambda x: x[1][1], reverse=True)[:top_n])

    for k in range(2, max_k+1):
        L[k] = {}
        prev = list(L[k-1].keys())
        candidates = set()

        for i in range(len(prev)):
            for j in range(i+1, len(prev)):
                union = prev[i] | prev[j]
                if len(union) == k:
                    candidates.add(union)

        pruned = []
        for c in candidates:
            if all((c - frozenset([x])) in L[k-1] for x in c):
                pruned.append(c)

        for c in pruned:
            items = list(c)
            bit = bit_data[frozenset([items[0]])]
            for item in items[1:]:
                bit &= bit_data[frozenset([item])]

            support = bin(bit).count('1') / total
            if support >= min_sup:
                L[k][c] = (bit, support)

        if len(L[k]) == 0:
            break

        L[k] = dict(sorted(L[k].items(), key=lambda x: x[1][1], reverse=True)[:top_n])

    results = []
    for k in L:
        for itemset, (_, sup) in L[k].items():
            results.append((tuple(itemset), sup))

    return sorted(results, key=lambda x: x[1], reverse=True)

# ==============================================
# COUNT
# ==============================================

def count_apriori(df, min_sup):
    return len(run_apriori(df, min_sup))

def count_fp(df, min_sup):
    return len(run_fp(df, min_sup))

def count_bitset(df, min_sup):
    return len(run_bitset_pruned_topN(df, min_sup))

# ==============================================
# 2.1 Algorithm Comparison
# ==============================================

print("\n===== 2.1 Algorithm Comparison =====")

df = datasets[1000]
min_sup = 0.05

(_, ap_time, ap_mem) = measure(lambda: run_apriori(df, min_sup))
(_, fp_time, fp_mem) = measure(lambda: run_fp(df, min_sup))
(_, bit_time, bit_mem) = measure(lambda: run_bitset_pruned_topN(df, min_sup))

print(f"Apriori → Time={ap_time:.4f}s | Memory={ap_mem:.2f}MB")
print(f"FP-Growth → Time={fp_time:.4f}s | Memory={fp_mem:.2f}MB")
print(f"Compact → Time={bit_time:.4f}s | Memory={bit_mem:.2f}MB")

# ==============================================
# 2.2.1 Scaling (WITH TIME + MEMORY)
# ==============================================

print("\n===== 2.2.1 Scaling =====")

for s in [1000, 3000, 5000, 10000]:
    df = datasets[s]

    (_, ap_time, ap_mem) = measure(lambda: count_apriori(df, min_sup))
    (_, fp_time, fp_mem) = measure(lambda: count_fp(df, min_sup))
    (_, bit_time, bit_mem) = measure(lambda: count_bitset(df, min_sup))

    print(f"\nN={s}")
    print(f"Apriori → Count={count_apriori(df, min_sup)} | Time={ap_time:.4f}s | Mem={ap_mem:.2f}MB")
    print(f"FP-Growth → Count={count_fp(df, min_sup)} | Time={fp_time:.4f}s | Mem={fp_mem:.2f}MB")
    print(f"Compact → Count={count_bitset(df, min_sup)} | Time={bit_time:.4f}s | Mem={bit_mem:.2f}MB")

# ==============================================
# 2.2.2 Min Support (WITH TIME + MEMORY)
# ==============================================

print("\n===== 2.2.2 Min Support =====")

min_sups = [0.01, 0.05, 0.1, 0.2]
df = datasets[1000]

for ms in min_sups:
    print(f"\n===== min_sup = {ms} =====")

    (_, ap_time, ap_mem) = measure(lambda: run_apriori(df, ms))
    (_, fp_time, fp_mem) = measure(lambda: run_fp(df, ms))
    (_, bit_time, bit_mem) = measure(lambda: run_bitset_pruned_topN(df, ms))

    fi_ap = run_apriori(df, ms)
    fi_ap = fi_ap[fi_ap['itemsets'].apply(lambda x: len(x) >= 2)]

    fi_fp = run_fp(df, ms)
    fi_fp = fi_fp[fi_fp['itemsets'].apply(lambda x: len(x) >= 2)]

    fi_bit = run_bitset_pruned_topN(df, ms)
    fi_bit = [(i, s) for i, s in fi_bit if len(i) >= 2]

    print(f"Apriori → Count={len(fi_ap)} | Time={ap_time:.4f}s | Mem={ap_mem:.2f}MB")
    print(f"FP-Growth → Count={len(fi_fp)} | Time={fp_time:.4f}s | Mem={fp_mem:.2f}MB")
    print(f"Compact → Count={len(fi_bit)} | Time={bit_time:.4f}s | Mem={bit_mem:.2f}MB")

# ==============================================
# 2.2.3 Max Length / Max K Experiment
# ==============================================

print("\n===== 2.2.3 Max Length / Max K =====")

df = datasets[1000]

for k in [4, 5, 6, 7, 8]:
    print(f"\n===== k = {k} =====")

    (_, ap_time, ap_mem) = measure(lambda: run_apriori(df, min_sup, max_len=k))
    (_, fp_time, fp_mem) = measure(lambda: run_fp(df, min_sup, max_len=k))
    (_, bit_time, bit_mem) = measure(lambda: run_bitset_pruned_topN(df, min_sup, max_k=k))

    ap_count = len(run_apriori(df, min_sup, max_len=k))
    fp_count = len(run_fp(df, min_sup, max_len=k))
    bit_count = len(run_bitset_pruned_topN(df, min_sup, max_k=k))

    print(f"Apriori → Count={ap_count} | Time={ap_time:.4f}s | Mem={ap_mem:.2f}MB")
    print(f"FP-Growth → Count={fp_count} | Time={fp_time:.4f}s | Mem={fp_mem:.2f}MB")
    print(f"Compact → Count={bit_count} | Time={bit_time:.4f}s | Mem={bit_mem:.2f}MB")


Loading 1000 transactions...
Loading 3000 transactions...
Loading 5000 transactions...
Loading 10000 transactions...

===== 2.1 Algorithm Comparison =====


/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Apriori → Time=0.0455s | Memory=3.21MB
FP-Growth → Time=0.0845s | Memory=0.47MB
Compact → Time=0.2844s | Memory=0.08MB

===== 2.2.1 Scaling =====


/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(



N=1000
Apriori → Count=90 | Time=0.0160s | Mem=3.17MB
FP-Growth → Count=90 | Time=0.0474s | Mem=0.47MB
Compact → Count=44 | Time=0.2728s | Mem=0.08MB


...

/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool type

Apriori → Count=90 | Time=0.0134s | Mem=3.17MB
FP-Growth → Count=90 | Time=0.0424s | Mem=0.47MB
Compact → Count=44 | Time=0.2482s | Mem=0.08MB

===== k = 7 =====


/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool type

Apriori → Count=90 | Time=0.0138s | Mem=3.17MB
FP-Growth → Count=90 | Time=0.0425s | Mem=0.47MB
Compact → Count=44 | Time=0.2419s | Mem=0.08MB

===== k = 8 =====
Apriori → Count=90 | Time=0.0135s | Mem=3.17MB
FP-Growth → Count=90 | Time=0.0426s | Mem=0.47MB
Compact → Count=44 | Time=0.2456s | Mem=0.08MB


/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
